# <u>MODELADO:</u>

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
df = pd.read_csv('../data/processed/model_cars.csv')

In [ ]:
df

,manufacturer,model,year,mileage,accidents_or_damage,one_owner,personal_use_only,seller_rating,driver_rating,driver_reviews_num,...,seller_rating_cat,driver_rating_cat,car_age,car_age_clip,mileage_per_year,mpg_mean,engine_power_index,log_price,log_mileage,log_driver_reviews_num
0,Acura,ILX Hybrid 1.5L,2013,47645.0,1,1,1,NaN,4.4,12,...,sin valoración,4-5,11,11,4331.363636,38.5,6.0,9.797849,10.771554,2.564949
1,Acura,ILX Hybrid 1.5L,2013,53422.0,0,1,1,4.3,4.4,12,...,4-5,4-5,11,11,4856.545455,38.5,6.0,9.740969,10.885997,2.564949
2,Acura,ILX Hybrid 1.5L,2013,117598.0,0,1,1,NaN,4.4,12,...,sin valoración,4-5,11,11,10690.727273,38.5,6.0,9.613002,11.675036,2.564949
3,Acura,ILX Hybrid 1.5L,2013,114865.0,1,0,1,3.7,4.4,12,...,3-4,4-5,11,11,10442.272727,38.5,6.0,9.581766,11.651522,2.564949
4,Acura,ILX Hybrid 1.5L,2013,62042.0,0,0,1,2.2,4.4,12,...,2-3,4-5,11,11,5640.181818,38.5,6.0,9.798127,11.035583,2.564949
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
591093,Volvo,S60 T5,2020,26781.0,1,1,1,5.0,4.9,24,...,4-5,4-5,4,4,6695.250000,28.5,8.0,10.337961,10.195485,3.218876
591094,Volvo,S60 B5 Momentum,2022,22877.0,0,1,0,4.2,4.2,2,...,4-5,4-5,2,2,11438.500000,29.0,8.0,10.457315,10.037931,1.098612
591095,Volvo,S60 T5,2014,92000.0,0,0,1,NaN,4.8,36,...,sin valoración,4-5,10,10,9200.000000,25.5,12.5,9.417273,11.429555,3.610918
591096,Volvo,S60 T5 Platinum,2013,132000.0,1,0,0,4.6,4.7,62,...,4-5,4-5,11,11,12000.000000,24.5,12.5,9.104424,11.790565,4.143135


## <u>1. Separación del dataset</u>

In [4]:
X = df.drop(columns=['price'])
y = df['price']

##### Train + temp (validación+test)

In [5]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42
)

##### Validación y test a partir de temp

In [6]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(X_train.shape, X_val.shape, X_test.shape)

(354658, 26) (118220, 26) (118220, 26)


##### Función de entrenamiento y evaluación del modelo

Para evitar duplicar código y facilitar la comparación entre distintos modelos,
se define una función que encapsula todo el proceso de:

- Preprocesado de variables numéricas y categóricas
- Entrenamiento del modelo de regresión
- Evaluación sobre el conjunto de validación

La función recibe como argumento la lista de variables numéricas, lo que permite
comparar fácilmente diferentes representaciones de una misma variable (por ejemplo,
`mileage` frente a `log_mileage`) manteniendo constante el resto del pipeline.


In [7]:
def train_and_evaluate(num_features, cat_features, X_train, y_train, X_val, y_val):

    num_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(
            handle_unknown='ignore',
            min_frequency=0.01
        ))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_transformer, num_features),
            ('cat', cat_transformer, cat_features)
        ]
    )

    model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', LinearRegression())
    ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    return mae, rmse

##### MODELO A (con mileage)

##### Definición de variables

In [8]:
num_features_mileage = [
    'car_age',
    'mileage',
    'mileage_per_year',
    'engine_liters',
    'engine_cylinders',
    'engine_power_index',
    'mpg_mean',
    'driver_reviews_num'
]

cat_features = [
    'manufacturer',
    'model',
    'fuel_simple',
    'transmission_simple',
    'drivetrain_simple',
    'accidents_or_damage',
    'one_owner',
    'personal_use_only',
    'seller_rating_cat',
    'driver_rating_cat'
]

##### Train + Validación 

In [9]:
mae_mileage, rmse_mileage = train_and_evaluate(
    num_features_mileage,
    cat_features,
    X_train, y_train,
    X_val, y_val
)

print("Modelo con mileage")
print(f"MAE: {mae_mileage:.2f}")
print(f"RMSE: {rmse_mileage:.2f}")

Modelo con mileage
MAE: 6398.01
RMSE: 10394.73


##### MODELO B (con log_mileage)

##### Definición de variables

In [10]:
num_features_log = [
    'car_age',
    'log_mileage',
    'mileage_per_year',
    'engine_liters',
    'engine_cylinders',
    'engine_power_index',
    'mpg_mean',
    'driver_reviews_num'
]

cat_features = [
    'manufacturer',
    'model',
    'fuel_simple',
    'transmission_simple',
    'drivetrain_simple',
    'accidents_or_damage',
    'one_owner',
    'personal_use_only',
    'seller_rating_cat',
    'driver_rating_cat'
]

##### Train + Validación

In [11]:
mae_log, rmse_log = train_and_evaluate(
    num_features_log,
    cat_features,
    X_train, y_train,
    X_val, y_val
)

print("Modelo con log_mileage")
print(f"MAE: {mae_log:.2f}")
print(f"RMSE: {rmse_log:.2f}")

Modelo con log_mileage
MAE: 6427.72
RMSE: 10439.70


##### Comparación final

In [12]:
results = pd.DataFrame({
    'Modelo': ['Mileage', 'Log mileage'],
    'MAE': [mae_mileage, mae_log],
    'RMSE': [rmse_mileage, rmse_log]
})

results

,Modelo,MAE,RMSE
0,Mileage,6398.009065,10394.730766
1,Log mileage,6427.718166,10439.700861


##### Conclusión de la comparación mileage vs log_mileage

Se entrenaron dos modelos idénticos, variando únicamente la forma de representar
el kilometraje del vehículo.

Los resultados muestran que el uso del kilometraje original (`mileage`) obtiene
ligeramente mejores valores de MAE y RMSE que la transformación logarítmica
(`log_mileage`).

Por tanto, en este caso, la transformación logarítmica no mejora el rendimiento
predictivo del modelo y se opta por mantener la variable `mileage` en su forma
original para los siguientes experimentos.


##### Se evalúa el mejor modelo (mileage) en el conjunto de test

In [13]:
best_num_features = num_features_mileage

mae_test, rmse_test = train_and_evaluate(
    best_num_features,
    cat_features,
    X_train, y_train,
    X_test, y_test
)

print(f"MAE test: {mae_test:.2f}")
print(f"RMSE test: {rmse_test:.2f}")

MAE test: 6413.75
RMSE test: 10428.28


### Decisión sobre el TARGET 

##### Modelo A: target = price

##### Modelo A (target=price). Es el mismo que el Modelo A (mileage) anterior

##### Modelo B: target = log_price

##### Se define el target transformado

In [14]:
y_log = df['log_price']

y_log_train = y_log.loc[X_train.index]
y_log_val   = y_log.loc[X_val.index]
y_log_test  = y_log.loc[X_test.index]

##### Definición del pipeline + Train


In [15]:
def train_model(num_features, cat_features, X_train, y_train):

    # Transformador numérico
    num_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    # Transformador categórico
    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(
            handle_unknown='ignore',
            min_frequency=0.01
        ))
    ])

    # Preprocesador completo
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_transformer, num_features),
            ('cat', cat_transformer, cat_features)
        ]
    )

    # Pipeline final
    model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', LinearRegression())
    ])

    # Entrenamiento
    model.fit(X_train, y_train)

    return model

##### Train

In [16]:
model_log_price = train_model(
    num_features_mileage,
    cat_features,
    X_train,
    y_log_train
)

##### Validación

In [17]:
y_log_pred_val = model_log_price.predict(X_val)
y_pred_val_logprice = np.exp(y_log_pred_val)

mae_logprice = mean_absolute_error(y_val, y_pred_val_logprice)
rmse_logprice = np.sqrt(mean_squared_error(y_val, y_pred_val_logprice))

print("Modelo con log_price (evaluado en €)")
print(f"MAE: {mae_logprice:.2f}")
print(f"RMSE: {rmse_logprice:.2f}")

Modelo con log_price (evaluado en €)
MAE: 5294.81
RMSE: 9611.90


##### Comparación 'price' vs 'log-price'

##### Preparación del modelo base (price) para la comparación

##### Alias para claridad semántica. (No hemos hecho el proceso de train + validation con price porque era exactamente lo mismo que con mileage que ya lo teníamos hecho de antes, por tanto, lo reutilizamos).

In [18]:
mae_price = mae_mileage
rmse_price = rmse_mileage

In [19]:
results_target = pd.DataFrame({
    'Target': ['price', 'log_price'],
    'MAE (€)': [mae_price, mae_logprice],
    'RMSE (€)': [rmse_price, rmse_logprice]
})

results_target

,Target,MAE (€),RMSE (€)
0,price,6398.009065,10394.730766
1,log_price,5294.812140,9611.897216


##### La transformación logarítmica del target mejora el rendimiento del modelo, reduciendo tanto el MAE como el RMSE. Por ello, se selecciona log_price como target final del modelo.


##### Se evalúa el mejor modelo (log_price) en el conjunto de test

In [21]:
y_log_pred_test = model_log_price.predict(X_test)
y_pred_test_logprice = np.exp(y_log_pred_test)

mae_test_logprice = mean_absolute_error(y_test, y_pred_test_logprice)
rmse_test_logprice = np.sqrt(mean_squared_error(y_test, y_pred_test_logprice))

print("Evaluación final en TEST (log_price, evaluado en €)")
print(f"MAE test: {mae_test_logprice:.2f}")
print(f"RMSE test: {rmse_test_logprice:.2f}")


Evaluación final en TEST (log_price, evaluado en €)
MAE test: 5316.24
RMSE test: 9647.68


##### Los resultados obtenidos en el conjunto de test son ligeramente superiores a los de validación, lo cual es esperable al tratarse de datos no utilizados durante el entrenamiento. La pequeña diferencia entre ambas métricas indica una buena capacidad de generalización del modelo y ausencia de overfitting.
